# Beat tracking under rubato: librosa vs. madmom

librosa's `beat_track` runs onset-strength + Ellis-style dynamic programming
against a single global tempo estimate -- it actively penalizes departure
from that tempo. madmom's RNN + dynamic-Bayesian-network pipeline allows
tempo to drift within learned bounds. The question: how much does each
tracker's F-measure degrade as a piece's rubato (tempo fluctuation)
increases, and does one degrade more gracefully than the other?

Ground truth comes from the [ASAP dataset](https://github.com/fosfrancesco/asap-dataset)
(score-aligned performances -- beat times come from alignment, not from
assuming a steady tempo), with audio for the MAESTRO-overlap subset. Scoring
is `mir_eval.beat.f_measure` (+/-70ms). See `thesis.md` for the full framing.

**This notebook runs in two modes:**
1. **Synthetic sanity check** (Section 1) -- always runs, no download needed.
   Confirms the environment and pipeline work end to end.
2. **Real ASAP/MAESTRO experiment** (Section 2 onward) -- needs the dataset
   built first:
   ```bash
   ./scripts/download_asap.sh
   ./scripts/download_maestro.sh            # ~103GB, asks for confirmation
   python scripts/build_asap_audio.py --maestro data/maestro-v2.0.0/maestro-v2.0.0
   ```
   Cells in Section 2+ detect whether this has been done and print
   instructions instead of failing if it hasn't.


In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
# import pandas as pd
import matplotlib.pyplot as plt
# import mir_eval
import soundfile as sf

# from src import datasets, rubato
from src.trackers import TRACKERS
# from src.evaluate import run_experiment

RESULTS_DIR = PROJECT_ROOT / "results"
ASAP_DIR = PROJECT_ROOT / "data" / "asap-dataset"
RESULTS_DIR.mkdir(exist_ok=True)

COLOR_LIBROSA = "#2a78d6"
COLOR_MADMOM = "#eb6834"


## Section 1: Synthetic sanity check

No dataset required. We synthesize click-track "performances" at increasing
levels of programmed rubato (a base tempo with a slow sinusoidal push/pull
applied to the beat spacing -- phrase-level, not per-beat jitter), run both
trackers, and score against the known ground truth. This exercises the exact
same code path (`src.trackers`, `src.rubato`, `src.evaluate`,
`mir_eval.beat.f_measure`) that Section 2 uses on real data.


In [7]:
SR = 22050
BASE_BPM = 100.0
N_BEATS = 120
RUBATO_LEVELS = {"none": 0.0, "mild": 0.08, "moderate": 0.18, "heavy": 0.30}


def make_click_track(rubato_depth: float, sr: int = SR):
    """Beat times with a sinusoidal tempo wobble, plus the rendered
    click-track audio. rubato_depth is the fractional swing in inter-beat
    interval (0 = perfectly steady tempo)."""
    base_ibi = 60.0 / BASE_BPM
    beat_idx = np.arange(N_BEATS)
    wobble = 1.0 + rubato_depth * np.sin(2 * np.pi * beat_idx / 16.0)
    ibi = base_ibi * wobble
    beat_times = np.concatenate([[0.5], 0.5 + np.cumsum(ibi)])

    duration = beat_times[-1] + 1.0
    audio = np.zeros(int(duration * sr))
    click_dur = 0.03
    t = np.arange(int(click_dur * sr)) / sr
    click = np.sin(2 * np.pi * 1200 * t) * np.exp(-t / 0.01)
    for bt in beat_times:
        start = int(bt * sr)
        end = min(start + len(click), len(audio))
        audio[start:end] += click[: end - start]

    audio = 0.8 * audio / (np.max(np.abs(audio)) + 1e-9)
    return beat_times, audio.astype(np.float32)


In [8]:
synthetic_dir = RESULTS_DIR / "smoke_test"
synthetic_dir.mkdir(exist_ok=True)

synthetic_performances = []
for level, depth in RUBATO_LEVELS.items():
    beat_times, audio = make_click_track(depth)

    wav_path = synthetic_dir / f"click_{level}.wav"
    sf.write(str(wav_path), audio, SR, subtype="PCM_16")  # madmom's reader needs int PCM

    ann_path = synthetic_dir / f"click_{level}_annotations.txt"
    with open(ann_path, "w") as f:
        for t in beat_times:
            f.write(f"{t}\t{t}\tb\n")

    synthetic_performances.append({
        "composer": "Synthetic", "title": f"click_{level}", "folder": str(synthetic_dir),
        "audio_path": wav_path, "annotation_path": ann_path,
    })

synthetic_df = run_experiment(
    asap_dir=synthetic_dir,
    output_csv=synthetic_dir / "smoke_results.csv",
    performances=synthetic_performances,
)
synthetic_df[["title", "ibi_std", "librosa_f_measure", "madmom_f_measure"]]


NameError: name 'run_experiment' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(synthetic_df["ibi_std"], synthetic_df["librosa_f_measure"], "o-", color=COLOR_LIBROSA, label="librosa")
ax.plot(synthetic_df["ibi_std"], synthetic_df["madmom_f_measure"], "o-", color=COLOR_MADMOM, label="madmom")
ax.set_xlabel("Rubato measure (ibi_std, synthetic)")
ax.set_ylabel("F-measure")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Synthetic sanity check: does the pipeline reproduce the expected shape?")
ax.legend(frameon=False)
fig.tight_layout()


If the environment is set up correctly, `librosa`'s F-measure should visibly
drop as `ibi_std` increases while `madmom`'s stays comparatively flat --
the qualitative pattern thesis.md predicts. This is a synthetic click
track, though, so treat it only as a pipeline check, not a finding.


## Section 2: Real ASAP/MAESTRO experiment

Requires the dataset to be built (see the setup commands in the intro cell).
The next cell checks and tells you what's missing rather than failing.


In [ ]:
asap_ready = (ASAP_DIR / "metadata.csv").exists()
n_available = datasets.count_available_audio(ASAP_DIR) if asap_ready else 0

if not asap_ready:
    print(f"ASAP dataset not found at {ASAP_DIR}.")
    print("Run: ./scripts/download_asap.sh")
elif n_available == 0:
    print(f"ASAP dataset found at {ASAP_DIR}, but no audio has been built yet.")
    print("Run: ./scripts/download_maestro.sh   (then) python scripts/build_asap_audio.py --maestro <path>")
else:
    print(f"Found {n_available} ASAP performances with built audio. Ready to run the real experiment.")


### Inspect the metadata

Composer/title/folder columns, plus which performances have audio available
locally right now.


In [6]:
if asap_ready:
    metadata = datasets.load_metadata(ASAP_DIR)
    display(metadata.head())
    print(f"{len(metadata)} performances total, {metadata['audio_performance'].notna().sum()} with a MAESTRO audio match.")


### Look at one performance: reference vs. estimated beats

Pick the first available performance and plot both trackers' beat estimates
against the ground truth, to sanity-check the trackers are producing
sensible output before running the full batch.


In [7]:
if n_available > 0:
    example = next(datasets.iter_performances_with_audio(ASAP_DIR))
    reference_beats = datasets.load_beat_times(example["annotation_path"])

    estimates = {name: track_fn(example["audio_path"]) for name, track_fn in TRACKERS.items()}

    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.vlines(reference_beats, 0, 1, color="#0b0b0b", linewidth=1, label="reference")
    ax.vlines(estimates["librosa"], 1, 2, color=COLOR_LIBROSA, linewidth=1, label="librosa")
    ax.vlines(estimates["madmom"], 2, 3, color=COLOR_MADMOM, linewidth=1, label="madmom")
    ax.set_yticks([])
    ax.set_xlabel("time (s)")
    ax.set_xlim(0, 30)  # first 30s -- zoom in, full pieces are long
    ax.set_title(f"{example['composer']} -- {example['title']} (first 30s)")
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout()


### Run the full batch evaluation

One performance per unique piece by default (`unique_pieces=True`), so
pieces with many recorded performances don't dominate. This can take a
while over hundreds of pieces since madmom's RNN pass is the bottleneck --
start with `max_pieces` set to something small to gauge runtime.


In [8]:
if n_available > 0:
    results_csv = RESULTS_DIR / "results.csv"
    results_df = run_experiment(
        asap_dir=ASAP_DIR,
        output_csv=results_csv,
        unique_pieces=True,
        max_pieces=None,  # set e.g. 20 for a quick partial run
    )
    display(results_df.describe())


### The finding: F-measure vs. rubato

The plot that turns this from a leaderboard into a finding (per thesis.md):
does F-measure fall off a cliff as rubato increases (librosa's predicted
failure mode), or bend gracefully (madmom's)?


In [9]:
if n_available > 0:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    from plot_results import plot as plot_f_measure_vs_rubato

    plot_f_measure_vs_rubato(results_df, "ibi_std", RESULTS_DIR / "f_measure_vs_ibi_std.png")
    plot_f_measure_vs_rubato(results_df, "ibi_cv", RESULTS_DIR / "f_measure_vs_ibi_cv.png")


## Notes / next steps

- `ibi_std` (raw std of inter-beat intervals) is the most literal reading of
  "rubato" but conflates with absolute tempo; `ibi_cv` normalizes for that.
  `local_tempo_wobble` further isolates beat-to-beat push/pull from smooth
  accelerando/ritardando arcs -- worth comparing all three if the raw
  `ibi_std` trend looks noisy.
- Why this matters beyond the leaderboard: a co-performance or
  score-following system is a real-time beat/position tracker pointed at a
  human playing expressively. If tracking collapses under rubato, so does
  the whole system -- this experiment probes exactly that failure mode.
